# Defense Mechanisms Against Inference Attacks

Three defenses are implemented:

| # | Defense | Description |
|---|---|---|
| 1 | **Gaussian Noise Injection** | Add N(0, σ²) to features before training |
| 2 | **Subject-Aware Noise** | Inject more noise on features correlated with subject identity |
| 3 | **Gradient Clipping + Noise** | Classic DP-SGD (manual, dependency-free) |

Each defense is parameterized by a single `noise_std` (or `epsilon` for DP), making ablation studies easy.

## 1. Imports & Device Setup

In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from typing import Dict, List, Tuple, Optional

from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import StandardScaler

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cpu


## 2. Defense 1 — Gaussian Noise Injection

Adds isotropic Gaussian noise to input features **before** the model sees them.  
Simple and parameter-free. Controlled by a single `noise_std` value.

In [2]:
class GaussianNoise:
    """
    Adds isotropic Gaussian noise to input features.
    Simple, parameter-free defense. Noise applied BEFORE the model.

    Args:
        noise_std: Standard deviation of Gaussian noise to add.
    """

    def __init__(self, noise_std: float = 0.1):
        self.noise_std = noise_std

    def apply(self, X: torch.Tensor) -> torch.Tensor:
        if self.noise_std <= 0:
            return X
        return X + torch.randn_like(X) * self.noise_std

    def __repr__(self):
        return f"GaussianNoise(sigma={self.noise_std})"

In [3]:
# Quick test
gn = GaussianNoise(noise_std=0.1)
x  = torch.randn(16, 561)
xn = gn.apply(x)
print(gn)
print(f"Input  std : {x.std():.4f}")
print(f"Output std : {xn.std():.4f}")

GaussianNoise(sigma=0.1)
Input  std : 1.0065
Output std : 1.0127


## 3. Defense 2 — Subject-Aware Noise Injection

Computes **mutual information** between each feature and subject identity.  
Features with high MI (strongly correlated with subject) get **more** noise.  
Features weakly correlated with subject get **less** noise.

This preserves activity-relevant features while masking identity-leaking ones — a better **privacy-utility tradeoff**.

In [4]:
class SubjectAwareNoise:
    """
    Per-feature noise scaled by MI(feature, subject identity).

    Workflow:
      1. Call .fit(X_train, y_subj) once to compute noise weights.
      2. Call .apply(X) or .apply_numpy(X) during training/inference.

    Args:
        base_std:     Base noise standard deviation.
        scale_factor: Max multiplier for high-MI features (range: [1, scale_factor]).
        n_features:   Expected feature dimensionality.
    """

    def __init__(
        self,
        base_std:     float = 0.1,
        scale_factor: float = 2.0,
        n_features:   int   = 561,
    ):
        self.base_std      = base_std
        self.scale_factor  = scale_factor
        self.noise_weights: Optional[np.ndarray] = None  # shape: (n_features,)
        self.fitted        = False

    def fit(
        self,
        X:       np.ndarray,   # (N, F)
        y_subj:  np.ndarray,   # (N,)
        verbose: bool = True,
    ) -> "SubjectAwareNoise":
        """
        Compute per-feature noise scale from MI(feature, subject).
        Call once before training with the full training set.
        """
        print("[defense] Computing mutual information (feature <-> subject)...")
        mi = mutual_info_classif(X, y_subj, random_state=42, n_jobs=-1)

        # Normalize MI to [1, scale_factor]
        mi_norm = (mi - mi.min()) / (mi.max() - mi.min() + 1e-8)
        self.noise_weights = 1.0 + (self.scale_factor - 1.0) * mi_norm

        if verbose:
            top10_idx = np.argsort(mi)[-10:][::-1]
            print(f"  Top-10 identity-leaking features : {top10_idx.tolist()}")
            print(f"  MI range          : {mi.min():.4f} - {mi.max():.4f}")
            print(f"  Noise scale range : {self.noise_weights.min():.2f} - "
                  f"{self.noise_weights.max():.2f}")

        self.fitted = True
        return self

    def apply(self, X: torch.Tensor) -> torch.Tensor:
        """Apply noise to a PyTorch tensor (for use inside training loops)."""
        if not self.fitted:
            raise RuntimeError("Call .fit() before .apply()")
        weights = torch.tensor(self.noise_weights, dtype=torch.float32, device=X.device)
        noise   = torch.randn_like(X) * self.base_std * weights
        return X + noise

    def apply_numpy(self, X: np.ndarray) -> np.ndarray:
        """Apply noise to a NumPy array."""
        noise = np.random.randn(*X.shape) * self.base_std * self.noise_weights
        return X + noise

    def __repr__(self):
        return (f"SubjectAwareNoise(base_std={self.base_std}, "
                f"scale={self.scale_factor})")

In [5]:
# Quick test with synthetic data: 300 samples, 561 features, 30 subjects
np.random.seed(42)
X_np   = np.random.randn(300, 561).astype(np.float32)
y_subj = np.repeat(np.arange(30), 10)   # 30 subjects x 10 samples each

san = SubjectAwareNoise(base_std=0.1, scale_factor=3.0)
san.fit(X_np, y_subj)

X_noisy = san.apply_numpy(X_np)
print(f"\n{san}")
print(f"Input  std : {X_np.std():.4f}")
print(f"Output std : {X_noisy.std():.4f}")

[defense] Computing mutual information (feature <-> subject)...
  Top-10 identity-leaking features : [557, 74, 558, 127, 498, 455, 538, 350, 527, 420]
  MI range          : 0.0000 - 0.1227
  Noise scale range : 1.00 - 3.00

SubjectAwareNoise(base_std=0.1, scale=3.0)
Input  std : 0.9995
Output std : 1.0076


## 4. Defense 3 — Manual DP-SGD (Gradient Clipping + Noise)

Implements differentially-private SGD following **Abadi et al. (2016)**, without external libraries.

**Algorithm per step:**
1. Compute per-sample gradient
2. Clip to L2 norm `C`
3. Add Gaussian noise `N(0, (σ·C)²)` to clipped gradients
4. Average and update

Privacy budget tracked via a simplified Rényi DP bound.

In [6]:
class DPSGDTrainer:
    """
    Differentially-private SGD without external libraries.

    Args:
        clip_norm:  C — per-sample gradient clipping norm.
        noise_mult: sigma — noise multiplier (larger = more privacy, less utility).
        delta:      Target delta for (epsilon, delta)-DP guarantee.
    """

    def __init__(
        self,
        clip_norm:  float = 1.0,
        noise_mult: float = 1.1,
        delta:      float = 1e-5,
    ):
        self.clip_norm  = clip_norm
        self.noise_mult = noise_mult
        self.delta      = delta
        self._steps     = 0
        self._n         = 0   # dataset size (set on first call)

    def privacy_spent(self, n_samples: int, batch_size: int) -> Dict:
        """
        Simplified epsilon estimate using Renyi DP moments accountant.
        Returns approximate (epsilon, delta) for training so far.

        Formula: epsilon ~= q * sqrt(2 * steps * log(1/delta)) / sigma
        where q = batch_size / n_samples (sampling rate).
        """
        q   = batch_size / n_samples
        eps = q * np.sqrt(2 * self._steps * np.log(1 / self.delta)) / self.noise_mult
        return {"epsilon": round(eps, 4), "delta": self.delta, "steps": self._steps}

    def train_step(
        self,
        model:     nn.Module,
        X_batch:   torch.Tensor,
        y_batch:   torch.Tensor,
        optimizer: torch.optim.Optimizer,
        criterion: nn.Module,
    ) -> float:
        """
        One DP-SGD step. Computes per-sample gradients manually.

        Returns:
            Mean loss for the batch.
        """
        model.train()
        batch_size = X_batch.size(0)

        # Accumulate clipped per-sample gradients
        accumulated_grads = [torch.zeros_like(p) for p in model.parameters()]
        total_loss = 0.0

        for xi, yi in zip(X_batch, y_batch):
            optimizer.zero_grad()
            out  = model(xi.unsqueeze(0))
            loss = criterion(out, yi.unsqueeze(0))
            loss.backward()
            total_loss += loss.item()

            # Clip per-sample gradient
            grads    = [p.grad.clone() for p in model.parameters() if p.grad is not None]
            grad_vec = torch.cat([g.view(-1) for g in grads])
            l2_norm  = grad_vec.norm(2)
            scale    = min(1.0, self.clip_norm / (l2_norm.item() + 1e-8))

            for accum, g in zip(accumulated_grads, grads):
                accum.add_(g * scale)

        # Add Gaussian noise to accumulated gradients
        for accum, p in zip(accumulated_grads, model.parameters()):
            if p.grad is not None:
                noise = torch.randn_like(accum) * self.clip_norm * self.noise_mult
                p.grad = (accum + noise) / batch_size

        optimizer.step()
        self._steps += 1
        return total_loss / batch_size

    def train_epoch(
        self,
        model:     nn.Module,
        loader:    DataLoader,
        optimizer: torch.optim.Optimizer,
        criterion: nn.Module = nn.CrossEntropyLoss(),
    ) -> float:
        """
        Run one full epoch of DP-SGD training.

        Returns:
            Mean loss over the epoch.
        """
        model = model.to(DEVICE)
        total_loss, n = 0.0, 0
        for X, y_act, _ in loader:
            X, y_act = X.to(DEVICE), y_act.to(DEVICE)
            loss = self.train_step(model, X, y_act, optimizer, criterion)
            total_loss += loss * len(X)
            n          += len(X)
        return total_loss / n

In [7]:
# Demonstrate privacy budget tracking
trainer = DPSGDTrainer(clip_norm=1.0, noise_mult=1.1, delta=1e-5)

# Simulate 100 steps
trainer._steps = 100
budget = trainer.privacy_spent(n_samples=1000, batch_size=32)

print("DP-SGD Privacy Budget after 100 steps:")
print(f"  epsilon : {budget['epsilon']}")
print(f"  delta   : {budget['delta']}")
print(f"  steps   : {budget['steps']}")

DP-SGD Privacy Budget after 100 steps:
  epsilon : 1.3959
  delta   : 1e-05
  steps   : 100


## 5. Defense Comparison Utility

Computes **privacy-utility tradeoff** metrics for one defense configuration.  
Useful for logging, comparison tables, and ablation plots.

In [8]:
def evaluate_defense(
    defense_name: str,
    noise_level:  float,
    act_acc:      float,
    asr:          float,
    baseline_asr: float,
) -> Dict:
    """
    Compute privacy-utility tradeoff metrics for one defense configuration.

    Args:
        defense_name: Human-readable name of the defense.
        noise_level:  Noise parameter used (e.g., noise_std or epsilon).
        act_acc:      Activity classification accuracy (utility metric).
        asr:          Attack Success Rate (privacy metric).
        baseline_asr: Majority-class ASR baseline (random guessing).

    Returns:
        Dict with privacy_gain, utility_cost, and a combined tradeoff_score.
        Higher tradeoff_score = better (more privacy retained, less utility lost).
    """
    privacy_gain = max(0, asr - baseline_asr)  # how much ASR is above random
    utility_cost = 1.0 - act_acc               # error rate

    # Combined score: 1.0 = perfect (ASR == baseline AND act_acc == 1)
    if privacy_gain + utility_cost < 1e-8:
        tradeoff_score = 1.0
    else:
        tradeoff_score = 1.0 - (privacy_gain / (privacy_gain + (1 - act_acc) + 1e-8))

    return {
        "defense":        defense_name,
        "noise_level":    noise_level,
        "act_acc":        round(act_acc, 4),
        "asr":            round(asr, 4),
        "baseline_asr":   round(baseline_asr, 4),
        "privacy_gain":   round(privacy_gain, 4),
        "utility_cost":   round(utility_cost, 4),
        "tradeoff_score": round(tradeoff_score, 4),
    }

In [9]:
# Example: compare three defense configurations
configurations = [
    ("No Defense",           0.0,  0.92, 0.75, 0.033),
    ("GaussianNoise",        0.1,  0.88, 0.55, 0.033),
    ("SubjectAwareNoise",    0.1,  0.90, 0.41, 0.033),
    ("DP-SGD",               1.1,  0.83, 0.38, 0.033),
]

results = [evaluate_defense(*cfg) for cfg in configurations]

print(f"{'Defense':<22} {'Noise':>6} {'ActAcc':>7} {'ASR':>6} {'Leak':>6} {'Tradeoff':>9}")
print("-" * 62)
for r in results:
    print(
        f"{r['defense']:<22} {r['noise_level']:>6.2f} "
        f"{r['act_acc']:>7.4f} {r['asr']:>6.4f} "
        f"{r['privacy_gain']:>6.4f} {r['tradeoff_score']:>9.4f}"
    )

Defense                 Noise  ActAcc    ASR   Leak  Tradeoff
--------------------------------------------------------------
No Defense               0.00  0.9200 0.7500 0.7170    0.1004
GaussianNoise            0.10  0.8800 0.5500 0.5170    0.1884
SubjectAwareNoise        0.10  0.9000 0.4100 0.3770    0.2096
DP-SGD                   1.10  0.8300 0.3800 0.3470    0.3288


## 6. Ablation: Noise Level vs Privacy-Utility Tradeoff

In [10]:
# Simulate ablation over noise_std values
# Replace with real act_acc / asr values from your experiments
noise_levels = [0.0, 0.05, 0.1, 0.2, 0.5, 1.0]

# Hypothetical values (substitute with real experiment results)
mock_act_acc = [0.92, 0.91, 0.88, 0.85, 0.75, 0.60]
mock_asr     = [0.75, 0.65, 0.55, 0.45, 0.20, 0.08]
baseline_asr = 0.033

ablation_results = [
    evaluate_defense("GaussianNoise", nl, aa, asr, baseline_asr)
    for nl, aa, asr in zip(noise_levels, mock_act_acc, mock_asr)
]

print(f"{'noise_std':>10} {'ActAcc':>8} {'ASR':>7} {'PrivacyLeak':>12} {'Tradeoff':>10}")
print("-" * 52)
for r in ablation_results:
    print(
        f"{r['noise_level']:>10.2f} {r['act_acc']:>8.4f} "
        f"{r['asr']:>7.4f} {r['privacy_gain']:>12.4f} {r['tradeoff_score']:>10.4f}"
    )

 noise_std   ActAcc     ASR  PrivacyLeak   Tradeoff
----------------------------------------------------
      0.00   0.9200  0.7500       0.7170     0.1004
      0.05   0.9100  0.6500       0.6170     0.1273
      0.10   0.8800  0.5500       0.5170     0.1884
      0.20   0.8500  0.4500       0.4170     0.2646
      0.50   0.7500  0.2000       0.1670     0.5995
      1.00   0.6000  0.0800       0.0470     0.8949


## 7. Full Pipeline Integration Example

Shows how to plug defenses into an existing training loop.

In [11]:
from torch.utils.data import TensorDataset, DataLoader

# Minimal MLP for demonstration
class SimpleMLP(nn.Module):
    def __init__(self, input_dim=561, hidden_dim=128, n_classes=6):
        super().__init__()
        self.fc1  = nn.Linear(input_dim, hidden_dim)
        self.fc2  = nn.Linear(hidden_dim, n_classes)
        self.relu = nn.ReLU()

    def extract_features(self, x):
        return self.relu(self.fc1(x))

    def forward(self, x):
        return self.fc2(self.extract_features(x))


def train_with_defense(
    model:     nn.Module,
    loader:    DataLoader,
    defense,                   # GaussianNoise | SubjectAwareNoise | None
    n_epochs:  int = 2,
    lr:        float = 1e-3,
) -> List[float]:
    """
    Standard training loop with optional input-space defense.
    For DP-SGD, use DPSGDTrainer.train_epoch() instead.
    """
    model     = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    losses    = []

    for epoch in range(n_epochs):
        epoch_loss, n = 0.0, 0
        model.train()
        for X, y_act, _ in loader:
            X, y_act = X.to(DEVICE), y_act.to(DEVICE)
            if defense is not None:
                X = defense.apply(X)
            optimizer.zero_grad()
            loss = criterion(model(X), y_act)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(X)
            n          += len(X)
        mean_loss = epoch_loss / n
        losses.append(mean_loss)
        print(f"  Epoch {epoch+1}/{n_epochs} — loss: {mean_loss:.4f}")

    return losses


# Synthetic data
torch.manual_seed(42)
X    = torch.randn(300, 561)
yact = torch.randint(0, 6, (300,))
ysub = torch.randint(0, 30, (300,))
dl   = DataLoader(TensorDataset(X, yact, ysub), batch_size=32)

In [12]:
# --- Train with no defense ---
print("=== No Defense ===")
model_plain = SimpleMLP()
train_with_defense(model_plain, dl, defense=None, n_epochs=2)

=== No Defense ===
  Epoch 1/2 — loss: 1.8122
  Epoch 2/2 — loss: 1.3456


[1.8121516656875611, 1.3455700095494587]

In [13]:
# --- Train with Gaussian noise defense ---
print("=== GaussianNoise (std=0.1) ===")
model_gn = SimpleMLP()
gn_defense = GaussianNoise(noise_std=0.1)
train_with_defense(model_gn, dl, defense=gn_defense, n_epochs=2)

=== GaussianNoise (std=0.1) ===
  Epoch 1/2 — loss: 1.8151
  Epoch 2/2 — loss: 1.3568


[1.8151198148727417, 1.3568143479029338]

In [14]:
# --- Train with Subject-Aware noise defense ---
print("=== SubjectAwareNoise (base_std=0.1, scale=3.0) ===")
model_san  = SimpleMLP()
san_defense = SubjectAwareNoise(base_std=0.1, scale_factor=3.0)
san_defense.fit(X.numpy(), ysub.numpy(), verbose=False)
train_with_defense(model_san, dl, defense=san_defense, n_epochs=2)

=== SubjectAwareNoise (base_std=0.1, scale=3.0) ===
[defense] Computing mutual information (feature <-> subject)...
  Epoch 1/2 — loss: 1.8061
  Epoch 2/2 — loss: 1.3479


[1.8061376269658407, 1.347892959912618]

In [15]:
# --- Train with DP-SGD ---
print("=== DP-SGD (clip=1.0, noise_mult=1.1) ===")
model_dp  = SimpleMLP().to(DEVICE)
optimizer = torch.optim.SGD(model_dp.parameters(), lr=1e-3)
dp_trainer = DPSGDTrainer(clip_norm=1.0, noise_mult=1.1, delta=1e-5)

for epoch in range(2):
    loss = dp_trainer.train_epoch(model_dp, dl, optimizer)
    budget = dp_trainer.privacy_spent(n_samples=300, batch_size=32)
    print(f"  Epoch {epoch+1} — loss: {loss:.4f}  |  "
          f"epsilon={budget['epsilon']}  delta={budget['delta']}")

=== DP-SGD (clip=1.0, noise_mult=1.1) ===
  Epoch 1 — loss: 1.8034  |  epsilon=1.4714  delta=1e-05
  Epoch 2 — loss: 1.8031  |  epsilon=2.0809  delta=1e-05


## 8. Summary

| Defense | Privacy Mechanism | Key Parameter | Notes |
|---|---|---|---|
| `GaussianNoise` | Isotropic input noise | `noise_std` | Fastest; uniform across features |
| `SubjectAwareNoise` | MI-weighted input noise | `base_std`, `scale_factor` | Better utility; requires fit step |
| `DPSGDTrainer` | Gradient clip + noise | `clip_norm`, `noise_mult` | Formal DP guarantee; slower training |